In [2]:
import torch
import numpy as np

# Create a simple tensor
x = torch.tensor([1.0, 2.0, 3.0, 4.0])

print("Tensor:", x)
print("Shape:", x.shape)
print("Dtype:", x.dtype)
print("Device:", x.device)

Tensor: tensor([1., 2., 3., 4.])
Shape: torch.Size([4])
Dtype: torch.float32
Device: cpu


In [3]:
# A 2D tensor — like a tiny grayscale image
image = torch.zeros(4, 4)
print("2D tensor:\n", image)
print("Shape:", image.shape)

# A 3D tensor — like a single microscopy image with 1 channel
image_3d = torch.zeros(1, 256, 256)
print("\n3D tensor shape:", image_3d.shape)

# A 4D tensor — like a batch of 8 single-channel images
batch = torch.zeros(8, 1, 256, 256)
print("4D tensor shape (batch):", batch.shape)


2D tensor:
 tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
Shape: torch.Size([4, 4])

3D tensor shape: torch.Size([1, 256, 256])
4D tensor shape (batch): torch.Size([8, 1, 256, 256])


In [ ]:

x = torch.zeros(8, 1, 512, 512)##Tensor is created on CPU by default
print("Before - Device:", x.device)

x = x.to("mps") ##Move x to the GPU (Metal Performance Shaders) 
print("After - Device:", x.device)
print("Shape unchanged:", x.shape)

Before - Device: cpu
After - Device: mps:0
Shape unchanged: torch.Size([8, 1, 512, 512])


In [6]:
# Create a tensor with gradient tracking enabled
x = torch.tensor(3.0, requires_grad=True)

# Do a simple mathematical operation
y = x ** 2

print("x:", x)
print("y = x²:", y)

# Compute gradients
y.backward()

# The gradient of y with respect to x
print("dy/dx:", x.grad)

x: tensor(3., requires_grad=True)
y = x²: tensor(9., grad_fn=<PowBackward0>)
dy/dx: tensor(6.)


In [7]:
# Simulate a tiny network: two learnable parameters
w = torch.tensor(2.0, requires_grad=True)  # a kernel value
b = torch.tensor(1.0, requires_grad=True)  # a bias

# Input data — a single pixel value
x = torch.tensor(4.0)

# Forward pass: simple linear operation
y_pred = w * x + b

# Loss: how far from the target?
y_true = torch.tensor(10.0)
loss = (y_pred - y_true) ** 2

print("Prediction:", y_pred.item())
print("Loss:", loss.item())

# Backpropagation
loss.backward()

print("\nGradient w.r.t. w:", w.grad.item())
print("Gradient w.r.t. b:", b.grad.item())

Prediction: 9.0
Loss: 1.0

Gradient w.r.t. w: -8.0
Gradient w.r.t. b: -2.0


In [8]:
learning_rate = 0.1

# Manual gradient descent update
with torch.no_grad():  # we don't want to track this operation
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad

# New prediction after one update
y_pred_new = w * x + b
loss_new = (y_pred_new - y_true) ** 2

print("Updated w:", w.item())
print("Updated b:", b.item())
print("New prediction:", y_pred_new.item())
print("New loss:", loss_new.item())
print("Old loss was 1.0 — did it decrease?")

Updated w: 2.799999952316284
Updated b: 1.2000000476837158
New prediction: 12.399999618530273
New loss: 5.759998321533203
Old loss was 1.0 — did it decrease?


In [11]:
import torch
import torch.nn as nn

# Every network in PyTorch is a class that inherits from nn.Module
class TinyNetwork(nn.Module):
    
    def __init__(self): ## define layers and parameters, this runs once you create the network
        super().__init__()
        # Define your layers here
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        ##in_channels=1: input has one channel,e.g. DAPI
        ##out_channels: 16 different features, 16 different kernels
        ##Padding: adds 1 pixel of zeros around the border of the image, so the output has the same spatial dimensions as the input
        ##stride (in the output size formula) is 1 by default, meaning the kernel moves one pixel at a time across the image
        self.relu  = nn.ReLU()
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=1, kernel_size=3, padding=1)

    def forward(self, x): ##define how data flows through the layers, This runs every time you feed an image
        # Define what happens when data flows through
        x = self.conv1(x)
        x = self.relu(x)
        x = self.conv2(x)
        return x

# Create the network
model = TinyNetwork()
print(model)

# Count how many learnable parameters it has
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal learnable parameters: {total_params}")


TinyNetwork(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu): ReLU()
  (conv2): Conv2d(16, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)

Total learnable parameters: 305


In [12]:
# Create a fake batch of 1 DAPI image, 256x256
fake_image = torch.zeros(1, 1, 256, 256)

# Forward pass
output = model(fake_image)

print("Input shape: ", fake_image.shape)
print("Output shape:", output.shape)

Input shape:  torch.Size([1, 1, 256, 256])
Output shape: torch.Size([1, 1, 256, 256])


output_size = (input_size + 2×padding - kernel_size) / stride + 1
input_size  = 256
padding     = 1
kernel_size = 3
stride      = 1

output = (256 + 2×1 - 3) / 1 + 1
       = (256 + 2 - 3) / 1 + 1
       = 255 / 1 + 1
       = 256  ✅
Spatial output stays 256x256 but we have 16 channels after conv1, then back to 1 channel after conv2.

In [ ]:
import torch
import torch.nn as nn

# 1 -- Create model and move to GPU
model = TinyNetwork().to("mps")

# 2 -- Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
"""Adam is an optimizer, alterative to stochastic gradient descent, 
adapts learning rate automatically for each parameter individually."""
# 3 -- Fake training data: 8 random images and targets
inputs  = torch.randn(8, 1, 256, 256).to("mps")
"""torch.randn creates a tensor filled with random numbers 
from a normal distribution (mean=0, std=1).rand creates normal random numbers"""
targets = torch.randn(8, 1, 256, 256).to("mps")

# 4 -- Training loop: 10 epochs
for epoch in range(10):
    
    # Forward pass
    predictions = model(inputs)
    
    # Compute loss
    loss = criterion(predictions, targets)
    
    # Zero gradients from previous step
    optimizer.zero_grad()
    """Pytorch accumulates gradients by default, so we need to zero them 
    out before each backward pass."""
    # Backward pass
    loss.backward()
    
    # Update parameters
    optimizer.step()
    
    print(f"Epoch {epoch+1}/10 - Loss: {loss.item():.4f}")

Epoch 1/10 - Loss: 1.1622
Epoch 2/10 - Loss: 1.1336
Epoch 3/10 - Loss: 1.1087
Epoch 4/10 - Loss: 1.0874
Epoch 5/10 - Loss: 1.0697
Epoch 6/10 - Loss: 1.0555
Epoch 7/10 - Loss: 1.0446
Epoch 8/10 - Loss: 1.0366
Epoch 9/10 - Loss: 1.0313
Epoch 10/10 - Loss: 1.0281
